## Setup (Dependencies & Environment)

**Python**: 3.12+

**Install**:

```bash
uv add openai python-dotenv ipykernel
uv sync
```

or

```bash
pip install openai python-dotenv ipykernel
```

**Environment**:

Set `OPENAI_API_KEY` in your shell or create a `.env` file in the same folder (this notebook calls `load_dotenv(override=True)`).

Example `.env`:

```bash
OPENAI_API_KEY=your_key_here
```


# Embeddings

This notebook demonstrates semantic closeness using cosine similarity.
Words with similar meanings should have higher similarity scores than unrelated ones.


In [1]:
from dotenv import load_dotenv

# Load environment variables
load_dotenv(override=True)

True

## Getting embeddings

In [2]:
from openai import OpenAI
client = OpenAI()

response = client.embeddings.create(
    input="Your text string goes here",
    model="text-embedding-3-small"
)

print(response.data[0].embedding)

[0.005126522853970528, 0.01728670485317707, -0.018700435757637024, -0.018574459478259087, -0.04722699895501137, -0.03029022552073002, 0.027686722576618195, 0.0036358069628477097, 0.011218861676752567, 0.006403779145330191, -0.0017304201610386372, 0.015816984698176384, -0.0013078757328912616, -0.007803512271493673, 0.059852588921785355, 0.05033440515398979, -0.027504757046699524, 0.00991710927337408, -0.04042429476976395, 0.04999846965074539, -0.00039804913103580475, 0.030234236270189285, -0.01373138278722763, 0.0328657366335392, 0.01730070263147354, 0.016810795292258263, -0.001747041940689087, 0.020422106608748436, 0.04076023027300835, -0.03776480257511139, -0.026133017614483833, -0.04997047409415245, 0.024187389761209488, -0.05526146665215492, -0.03230584040284157, 0.04235592484474182, 0.06461168080568314, 0.014683200977742672, -0.015635019168257713, -0.04134811833500862, 0.02217177301645279, 0.007362596690654755, 0.04490343853831291, 0.007089648395776749, -0.02406141348183155, 0.0524

In [3]:
# Dimension
len(response.data[0].embedding)

1536

In [4]:
response = client.embeddings.create(
    input="Your text string goes here",
    model="text-embedding-3-small",
    dimensions=100
)

print(response.data[0].embedding)
print(len(response.data[0].embedding))

[0.017870279029011726, 0.06025882437825203, -0.06518687307834625, -0.06474774330854416, -0.1646261364221573, -0.10558711737394333, 0.09651169925928116, 0.012673869729042053, 0.039107244461774826, 0.022322600707411766, -0.006031981203705072, 0.05513560399413109, -0.004559055902063847, -0.027201857417821884, 0.2086370289325714, 0.17545807361602783, -0.09587740153074265, 0.03456953540444374, -0.14091293513774872, 0.17428706586360931, -0.001387538737617433, 0.10539194941520691, -0.04786550998687744, 0.11456495523452759, 0.060307614505290985, 0.05859987810254097, -0.0060899225063622, 0.07118836045265198, 0.14208395779132843, -0.13164235651493073, -0.09109573066234589, -0.1741894781589508, 0.08431356400251389, -0.19263306260108948, -0.1126132532954216, 0.14764632284641266, 0.2252265065908432, 0.05118340626358986, -0.05450129881501198, -0.14413325488567352, 0.07728742808103561, 0.02566489204764366, 0.1565265655517578, 0.024713436141610146, -0.0838744267821312, 0.18277697265148163, -0.07006613

## Cosine similarity

Cosine similarity measures how close two vectors are in direction (ignoring magnitude).
A score of **1.0** means identical direction, **0** means orthogonal, and **−1** means opposite.

In [5]:
import math

def cosine_similarity(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    norm_a = math.sqrt(sum(x * x for x in a))
    norm_b = math.sqrt(sum(x * x for x in b))
    return dot / (norm_a * norm_b) if norm_a and norm_b else 0.0

In [6]:
# Quick sanity check with simple 2D vectors
cosine_similarity([1, 0], [1, 0])
# cosine_similarity([1, 0], [-1, 0])
#cosine_similarity([1, 0], [0, 1])
#cosine_similarity([1, 0], [1, 1])

1.0

## Comparing word embeddings

Let's embed a set of words and see how cosine similarity captures semantic relationships.

In [7]:
words = [
    "cat", "dog", "kitten", "puppy",
    "car", "automobile",
    "banana", "apple",
    "king", "queen", "man", "woman",
    "Paris", "France", "Tokyo", "Japan", "Singapore",
]

response = client.embeddings.create(
    model="text-embedding-3-small",
    input=words,
)

embedding_by_word = {
    word: response.data[i].embedding
    for i, word in enumerate(words)
}

In [8]:
pairs = [
    ("cat", "dog"),
    ("kitten", "puppy"),
    ("car", "automobile"),
    ("king", "queen"),
    ("man", "woman"),
    ("Paris", "France"),
    ("Paris", "Singapore"),
    ("Tokyo", "Japan"),
    ("banana", "apple"),
    ("cat", "banana"),
    ("car", "king"),
    ("Paris", "puppy"),
    ("queen", "automobile"),
]

print("Cosine similarity (higher = more similar):\n")
for a, b in pairs:
    score = cosine_similarity(embedding_by_word[a], embedding_by_word[b])
    print(f"  {a:<12} {b:<12} {score:.3f}")

Cosine similarity (higher = more similar):

  cat          dog          0.603
  kitten       puppy        0.650
  car          automobile   0.556
  king         queen        0.591
  man          woman        0.707
  Paris        France       0.661
  Paris        Singapore    0.399
  Tokyo        Japan        0.471
  banana       apple        0.476
  cat          banana       0.309
  car          king         0.331
  Paris        puppy        0.204
  queen        automobile   0.219
